# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2k24csaiml1e2411265-wq/ML_starter_FlyrankAI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Answer:** One row represents one **client-content item pair** for a defined decision window. I use **February 2026** as the feature window and **March 2026** as the outcome window. The decision moment is 1 March 2026. I use this mid-panel period for development and avoid the final-month `_sample` table for label development.

In [14]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

import os, getpass, duckdb, pandas as pd
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('')
assert HF_TOKEN, 'HF_TOKEN is required.'
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print('DuckDB connected to FlyRank warehouse.')

··········
DuckDB connected to FlyRank warehouse.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features:** `imp_prev30`, `clk_prev30`, `ctr_prev30`, `pos_prev30`, `query_concentration`.

**Label:** `is_declining` — March 2026 impressions are more than 20% below the previous 30-day impressions.

**Context:** `client_hash_id`, `content_hash_id`, and date-window information used to define and join the panel.

**Excluded:** future-window impressions, clicks, CTR, position, and any outcome-derived value. These are excluded because they would not be known at the decision moment and would leak the answer.

In [15]:
schema_daily = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']}").df()
print('Daily fact columns:')
display(schema_daily)

Daily fact columns:


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The three verification queries below check **grain**, **March 2026 count/date span**, and **availability using `IS TRUE` when a boolean availability field is exposed**.

In [16]:
# QUERY 1 — Grain: one row should be unique at client + content + date.
q1 = con.sql(f"""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR)) AS distinct_grain_keys,
       COUNT(*) - COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR)) AS duplicate_rows
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
""").df()
display(q1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_keys,duplicate_rows
0,9841378,9841378,0


In [17]:
# QUERY 2 — March 2026 row count and observed date span.
q2 = con.sql(f"""
SELECT COUNT(*) AS row_count,
       MIN(report_date) AS min_report_date,
       MAX(report_date) AS max_report_date,
       COUNT(DISTINCT client_hash_id) AS clients,
       COUNT(DISTINCT content_hash_id) AS content_items
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
""").df()
display(q2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_report_date,max_report_date,clients,content_items
0,9841378,2026-03-01,2026-03-31,55,331437


In [18]:
# QUERY 3 — Availability/missingness. Use IS TRUE if a boolean availability field exists.
bool_cols = [r['column_name'] for _, r in schema_daily.iterrows() if str(r['column_type']).upper() in ('BOOLEAN','BOOL')]
preferred = [c for c in bool_cols if any(k in c.lower() for k in ['available','valid','observed'])]
availability_col = preferred[0] if preferred else (bool_cols[0] if bool_cols else None)
print('Boolean columns:', bool_cols)
print('Availability column used:', availability_col)
if availability_col:
    q3 = con.sql(f"""
    SELECT COUNT(*) AS march_rows,
           COUNT(*) FILTER (WHERE {availability_col} IS TRUE) AS available_rows,
           COUNT(*) FILTER (WHERE {availability_col} IS NOT TRUE) AS unavailable_or_null_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
    """).df()
else:
    q3 = con.sql(f"""
    SELECT COUNT(*) AS march_rows,
           COUNT(*) FILTER (WHERE gsc_impressions IS NOT NULL) AS rows_with_impressions,
           COUNT(*) FILTER (WHERE gsc_clicks IS NOT NULL) AS rows_with_clicks,
           COUNT(*) FILTER (WHERE gsc_avg_position IS NOT NULL) AS rows_with_position
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
    """).df()
    print('No boolean availability field was exposed; core-field availability is shown instead.')
display(q3)

Boolean columns: ['client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available']
Availability column used: gsc_data_available


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,available_rows,unavailable_or_null_rows
0,9841378,3611061,6230317


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Limitation:** This is an **unbalanced panel**, so clients have different history depth and a 30-day feature window may not be available for every client-content pair. GSC data measures observed search behavior but cannot by itself establish why a page declined or prove that a refresh caused improvement. Query-mix signals are directional because rare/anonymized queries limit interpretation of individual search terms.

I therefore treat these features as **decision-support signals**, not causal evidence, and would use minimum-history rules plus client-level evaluation before claiming generalization.

In [19]:
# Five-feature frame. Heavy aggregation happens in DuckDB; only the small result enters pandas.
features = con.sql(f"""
WITH base AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_prev30,
           SUM(gsc_clicks) AS clk_prev30,
           AVG(gsc_avg_position) AS pos_prev30
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-02-01' AND report_date < DATE '2026-03-01'
    GROUP BY 1,2
    HAVING SUM(gsc_impressions) >= 100
), queries AS (
    SELECT content_hash_id,
           SUM(impressions_90d) AS kept_impressions,
           MAX(impressions_90d) AS top_query_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
)
SELECT b.client_hash_id, b.content_hash_id, b.imp_prev30, b.clk_prev30,
       CASE WHEN b.imp_prev30 > 0 THEN b.clk_prev30 / b.imp_prev30 ELSE NULL END AS ctr_prev30,
       b.pos_prev30,
       CASE WHEN q.kept_impressions > 0 THEN q.top_query_impressions / q.kept_impressions ELSE NULL END AS query_concentration
FROM base b LEFT JOIN queries q USING (content_hash_id)
""").df()
print(f'Feature rows: {len(features):,}')
display(features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 80,322


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,ctr_prev30,pos_prev30,query_concentration
0,client_08a6a72ff48e62c0,content_5f6fae04728d32ab,324.0,0.0,0.000000,5.337775,0.377622
1,client_08a6a72ff48e62c0,content_5f71205e0b46f70a,1574.0,3.0,0.001906,2.909252,0.306488
2,client_08a6a72ff48e62c0,content_5f716493f7989b45,510.0,2.0,0.003922,7.437261,0.213628
3,client_08a6a72ff48e62c0,content_5fb5bd16c7ad75f1,107.0,0.0,0.000000,37.262363,0.586081
4,client_08a6a72ff48e62c0,content_5fbc3a12b51e755f,156.0,0.0,0.000000,64.912776,0.485714


### Five features — “available when?”

1. **`imp_prev30`** — known after the previous 30 days are observed, before the March outcome window.
2. **`clk_prev30`** — known after the previous 30 days are observed, before the March outcome window.
3. **`ctr_prev30`** — calculated only from previous-window clicks and impressions.
4. **`pos_prev30`** — average search position from the previous window only.
5. **`query_concentration`** — historical query-mix context from the 90-day query table; treated as directional.

### Deliberate leakage check

The next cells create the March outcome label and intentionally add one label-derived feature. The leaked score should look unrealistically strong because the model is being given information from the answer itself. The leaked feature is then removed.

In [20]:
# March outcome label: >20% impression decline versus the previous 30 days.
outcome = con.sql(f"""
SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_future30
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
GROUP BY 1,2
""").df()
data = features.merge(outcome, on=['client_hash_id','content_hash_id'], how='inner')
data['is_declining'] = (data['imp_future30'] < 0.8 * data['imp_prev30']).astype(int)
print(f'Rows with feature + outcome windows: {len(data):,}')
print(f'Decline rate: {data["is_declining"].mean():.3f}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows with feature + outcome windows: 76,837
Decline rate: 0.182


In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
feature_cols = ['imp_prev30','clk_prev30','ctr_prev30','pos_prev30','query_concentration']
model_data = data.dropna(subset=feature_cols + ['is_declining']).copy()
X, y = model_data[feature_cols], model_data['is_declining']
X_tr, X_te, y_tr, y_te = train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)
honest_model = RandomForestClassifier(n_estimators=150,random_state=42,n_jobs=-1).fit(X_tr,y_tr)
honest_pred = honest_model.predict(X_te)
print('HONEST MODEL')
print('Accuracy:', round(accuracy_score(y_te,honest_pred),3))
print(classification_report(y_te,honest_pred,digits=3))

HONEST MODEL
Accuracy: 0.839
              precision    recall  f1-score   support

           0      0.853     0.979     0.912     13972
           1      0.321     0.057     0.096      2494

    accuracy                          0.839     16466
   macro avg      0.587     0.518     0.504     16466
weighted avg      0.773     0.839     0.788     16466



In [22]:
# INTENTIONAL LEAK: this feature directly contains the label.
model_data['leak_future_decline'] = model_data['is_declining']
X_leak = model_data[feature_cols + ['leak_future_decline']]
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(X_leak,y,test_size=0.25,random_state=42,stratify=y)
leak_model = RandomForestClassifier(n_estimators=150,random_state=42,n_jobs=-1).fit(Xl_tr,yl_tr)
leak_pred = leak_model.predict(Xl_te)
print('LEAKED MODEL — intentionally invalid')
print('Accuracy:', round(accuracy_score(yl_te,leak_pred),3))
print(classification_report(yl_te,leak_pred,digits=3))

LEAKED MODEL — intentionally invalid
Accuracy: 1.0
              precision    recall  f1-score   support

           0      1.000     1.000     1.000     13972
           1      1.000     1.000     1.000      2494

    accuracy                          1.000     16466
   macro avg      1.000     1.000     1.000     16466
weighted avg      1.000     1.000     1.000     16466



### Leakage conclusion

The leaked model receives information that directly identifies the outcome, so its high score is not trustworthy. `leak_future_decline` is removed and is not part of the retained feature set. The honest feature set contains only information available at the decision moment.

In [23]:
model_data = model_data.drop(columns=['leak_future_decline'])
print('Final retained features:', feature_cols)
print('Leakage feature retained:', 'leak_future_decline' in model_data.columns)

Final retained features: ['imp_prev30', 'clk_prev30', 'ctr_prev30', 'pos_prev30', 'query_concentration']
Leakage feature retained: False


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.